# Gaussian Processes — Problema 1
## Exemplo 4.3: identificando uma função escalar com poucos dados

**Referência:** Sivaranjani et al., *Control-Oriented System Identification: Classical, Learning, and Physics-Informed Approaches*, Seção 4.2.5, Exemplo 4.3 e Figura 6 ([artigo](https://arxiv.org/html/2512.06315)).  
**Código de referência:** [notebook Structured_GP dos autores](https://github.com/ExistentialRobotics/physics_based_sysid/blob/main/Structured_GP/structured_GP.ipynb).

A ideia é reproduzir o problema **entendendo cada equação**. Primeiro fazemos um GP convencional; depois, incorporamos conhecimento da estrutura linear por partes à **média** do processo gaussiano.

**O que vem do artigo:** função verdadeira, 15 observações, comparação entre GP convencional e GP estruturado.  
**O que vem do notebook dos autores:** amostragem uniforme em [0, 2], semente 1, ruído gaussiano de desvio-padrão 0,1 e kernel Matérn.  
**Nossa escolha didática:** Matérn 3/2 explícito, implementação NumPy/SciPy e estimação de hiperparâmetros por máxima verossimilhança marginal. O original usa GPax e MCMC; esta é uma **reprodução independente**, não uma réplica numérica exata.

Execute as células na ordem. Dependências: NumPy, SciPy e Matplotlib. Este primeiro problema é **regressão estática**; o Duffing vem no notebook seguinte.


### 1. Enunciado do problema

Considere um sistema escalar estático em que a saída $y$ depende de uma entrada $x$ por uma função desconhecida $f:\mathbb{R}\rightarrow\mathbb{R}$.

Nosso objetivo é **estimar a função desconhecida a partir de observações ruidosas**, utilizando inicialmente a regressão por processos gaussianos (GPR) convencional e, depois, um GP com estrutura incorporada à média.

Para produzir um experimento reprodutível, a planta simulada é definida por

$$
f_{\mathrm{real}}(x)=\begin{cases}
ax, & x\leq x_{\mathrm{th}},\\
bx, & x>x_{\mathrm{th}},
\end{cases}
\qquad a=1,\quad b=-1,\quad x_{\mathrm{th}}=1.
$$

As observações são dadas por

$$
y_i=f_{\mathrm{real}}(x_i)+\varepsilon_i,
\qquad \varepsilon_i\overset{\mathrm{iid}}{\sim}\mathcal N(0,\sigma_n^2),
\qquad \sigma_n=0.1.
$$

Serão coletadas $N=15$ observações com entradas amostradas uniformemente em $[0,2]$:

$$
x_i\overset{\mathrm{iid}}{\sim}\mathcal U(0,2), \qquad
\mathcal D=\{(x_i,y_i)\}_{i=1}^{15}.
$$

**Separação fundamental:** a função $f_{\mathrm{real}}$ é conhecida **somente pelo simulador**, para gerar os dados e avaliar posteriormente as predições. O identificador receberá apenas $\mathcal D$; ele não receberá a fórmula da planta nem os valores de $a$, $b$ e $x_{\mathrm{th}}$.

> Nota: o Exemplo 4.3 do artigo define a função e os 15 pares observados. O intervalo $[0,2]$, o ruído com desvio-padrão $0.1$ e a semente 1 aparecem no notebook dos autores. O problema é de regressão estática; a identificação de dinâmica temporal virá depois, com o sistema Duffing.

## 2. Dados: o simulador sabe a verdade; o identificador, não

Os dados seguem
\[
x_i\overset{\mathrm{iid}}{\sim}\mathcal U(0,2),\qquad
y_i=f_{\mathrm{real}}(x_i)+\varepsilon_i,\qquad
\varepsilon_i\overset{\mathrm{iid}}{\sim}\mathcal N(0,0.1^2).
\]

A função verdadeira aparece apenas na simulação e na avaliação final. Os parâmetros reais **não serão usados pelo ajuste**.

Uma primeira dificuldade: a função tem um salto em \(x=1\), enquanto as funções amostradas de kernels Matérn usuais são contínuas.


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.linalg import cho_factor, cho_solve
from scipy.optimize import minimize

np.set_printoptions(precision=4, suppress=True)
N, SIGMA_RUIDO = 15, 0.1
A_REAL, B_REAL, LIMIAR_REAL = 1.0, -1.0, 1.0

def f_real(x):
    """Conhecida APENAS pelo simulador e pela avaliação."""
    x = np.asarray(x, dtype=float)
    return np.where(x <= LIMIAR_REAL, A_REAL*x, B_REAL*x)

rng = np.random.RandomState(1)  # reproduz a sequência NumPy do código dos autores
X_train = rng.uniform(0, 2, N)
y_train = f_real(X_train) + rng.normal(0, SIGMA_RUIDO, N)
X_test = np.linspace(0, 2, 501)

print("Quantidade de dados:", len(X_train))
print("Pontos observados (ordenados):", np.sort(X_train))


In [ ]:
fig, ax = plt.subplots(figsize=(9, 4))
ax.scatter(X_train, y_train, c="black", marker="x", s=65, label="observações ruidosas")
ax.set(xlabel="x", ylabel="y", title="Problema 1 — o que o identificador observa", xlim=(0,2))
ax.grid(alpha=.25); ax.legend()
plt.show()


## 3. Prior: função de média e kernel

Assumimos inicialmente \(m(x)=0\), isto é, **expectativa a priori nula**, não a afirmação de que a função real é nula.

Escolhemos o Matérn \(3/2\):
\[
k_\theta(x,x')=\sigma_f^2(1+r)e^{-r},\qquad
r=\frac{\sqrt{3}|x-x'|}{\ell}.
\]

Aqui, \(\ell>0\) controla a escala de correlação, \(\sigma_f>0\) a amplitude das flutuações da função e \(\sigma_n>0\) o ruído das **observações**. Não são três nomes para a mesma incerteza.

Por definição, \(f(X)\sim\mathcal N(m(X),K_{XX})\), onde \((K_{XX})_{ij}=k(x_i,x_j)\).


In [ ]:
def matern32(x, z, sigma_f, ell):
    x = np.asarray(x, dtype=float).reshape(-1, 1)
    z = np.asarray(z, dtype=float).reshape(1, -1)
    r = np.sqrt(3) * np.abs(x-z) / ell
    return sigma_f**2 * (1+r) * np.exp(-r)

# Controle didático: experimente alterar a escala de comprimento.
ELL_PRIOR = 0.35
SIGMA_F_PRIOR = 1.0
X_prior = np.linspace(0, 2, 140)
K_prior = matern32(X_prior, X_prior, SIGMA_F_PRIOR, ELL_PRIOR)
L = np.linalg.cholesky(K_prior + 1e-9*np.eye(len(X_prior)))
amostras = L @ np.random.default_rng(156).normal(size=(len(X_prior), 4))

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
for j in range(4):
    axes[0].plot(X_prior, amostras[:, j], lw=1)
axes[0].axhline(0, color="black", linestyle="--", label="m(x)=0")
axes[0].set(xlabel="x", ylabel="f(x)", title="Quatro funções amostradas do prior")
axes[0].legend()
im = axes[1].imshow(K_prior, extent=[0,2,0,2], origin="lower")
axes[1].set(xlabel="x'", ylabel="x", title="Matriz de covariância")
fig.colorbar(im, ax=axes[1])
plt.tight_layout(); plt.show()
print("Simetria:", np.allclose(K_prior, K_prior.T))
print("Menor autovalor (aprox.):", np.linalg.eigvalsh(K_prior)[0])


## 4. Condicionamento gaussiano: o posterior

A gaussiana conjunta das observações \(y\) e dos valores latentes \(f_*\) tem covariância
\[
\begin{bmatrix}
K_{XX}+\sigma_n^2I & K_{X*}\\
K_{*X} & K_{**}
\end{bmatrix}.
\]

Condicionar aos dados produz
\[
\boxed{\mu_*=m(X_*)+K_{*X}(K_{XX}+\sigma_n^2 I)^{-1}(y-m(X))}
\]
\[
\boxed{\Sigma_*=K_{**}-K_{*X}(K_{XX}+\sigma_n^2 I)^{-1}K_{X*}}.
\]

**Interpretação:** partimos de \(m(X_*)\) e aprendemos uma correção a partir do resíduo \(y-m(X)\). A incerteza de \(f_*\) é diferente da incerteza de uma **nova observação** \(y_*\), que adicionaria \(\sigma_n^2I\).

Usaremos fatoração de Cholesky, **nunca a inversa matricial explícita**.


In [ ]:
def media_zero(x):
    return np.zeros_like(np.asarray(x, dtype=float).reshape(-1))

def gp_predict(X, y, X_star, params, mean_fn=media_zero):
    sigma_f, ell, sigma_n = params
    K = matern32(X, X, sigma_f, ell)
    K[np.diag_indices_from(K)] += sigma_n**2 + 1e-9
    cf = cho_factor(K, lower=True, check_finite=False)
    r = y - mean_fn(X)
    alpha = cho_solve(cf, r, check_finite=False)
    Ks = matern32(X, X_star, sigma_f, ell)
    mu = mean_fn(X_star) + Ks.T @ alpha
    V = cho_solve(cf, Ks, check_finite=False)
    var = np.maximum(0, sigma_f**2 - np.sum(Ks*V, axis=0))
    return mu, np.sqrt(var)

# Primeiro posterior, com hiperparâmetros arbitrários FIXADOS.
PARAMETROS_INICIAIS = (1.0, 0.35, 0.1)
mu_fix, sd_fix = gp_predict(X_train, y_train, X_test, PARAMETROS_INICIAIS)
fig, ax = plt.subplots(figsize=(9, 4))
ax.scatter(X_train, y_train, c="black", marker="x", label="dados")
ax.plot(X_test, mu_fix, label="média posterior")
ax.fill_between(X_test, mu_fix-1.96*sd_fix, mu_fix+1.96*sd_fix, alpha=.2,
                label="±1,96 desvio-padrão (função latente)")
ax.set(xlabel="x", ylabel="f(x)", title="Posterior: hiperparâmetros fixados")
ax.legend(); ax.grid(alpha=.2); plt.show()


## 5. Sensibilidade dos hiperparâmetros

Antes de estimá-los, vamos observá-los separadamente. Uma escala de comprimento pequena permite mudanças rápidas de \(f\); uma escala grande impõe correlação sobre distâncias maiores. O ruído controla o quanto as observações devem ser acompanhadas.

**Experimento:** edite os três números, execute e tente antecipar o que ocorrerá com a média e a faixa posterior.


In [ ]:
ELL = 0.15       # escala de comprimento
SIGMA_F = 1.0    # amplitude do processo
SIGMA_N = 0.10   # desvio-padrão do ruído
mu_ctrl, sd_ctrl = gp_predict(X_train, y_train, X_test, (SIGMA_F, ELL, SIGMA_N))
plt.figure(figsize=(9, 4))
plt.scatter(X_train, y_train, c="black", marker="x", label="dados")
plt.plot(X_test, mu_ctrl, label="média posterior")
plt.fill_between(X_test, mu_ctrl-1.96*sd_ctrl, mu_ctrl+1.96*sd_ctrl, alpha=.2)
plt.xlabel("x"); plt.ylabel("f(x)")
plt.title(f"GP com ell={ELL}, sigma_f={SIGMA_F}, sigma_n={SIGMA_N}")
plt.legend(); plt.grid(alpha=.2); plt.show()


## 6. Verossimilhança marginal

No GP convencional, estimamos \(\theta=(\sigma_f,\ell,\sigma_n)\) minimizando a **negativa** da log-verossimilhança marginal:
\[
-\log p(y\mid X,\theta)
=\tfrac12 r^\top C^{-1}r+\tfrac12\log|C|+\tfrac n2\log(2\pi),
\quad r=y-m(X),\quad C=K_\theta(X,X)+\sigma_n^2I.
\]

Os três termos correspondem a ajuste dos dados, complexidade implícita do modelo e normalização probabilística.

Isso fornece um **estimador pontual** dos hiperparâmetros (empirical Bayes). Não é a integração da incerteza sobre hiperparâmetros realizada pelo MCMC do notebook original.


In [ ]:
def neg_log_marginal(log_params, X, y, mean_fn=media_zero):
    sf, ell, sn = np.exp(log_params)  # positividade garantida
    K = matern32(X, X, sf, ell)
    K[np.diag_indices_from(K)] += sn**2 + 1e-9
    try:
        cf = cho_factor(K, lower=True, check_finite=False)
        residual = y-mean_fn(X)
        alpha = cho_solve(cf, residual, check_finite=False)
        return float(0.5*residual@alpha + np.log(np.diag(cf[0])).sum()
                     + len(y)/2*np.log(2*np.pi))
    except np.linalg.LinAlgError:
        return 1e12

BOUNDS = [(np.log(.02), np.log(10.0)),
          (np.log(.025), np.log(5.0)),
          (np.log(.005), np.log(.6))]
ajuste_conv = minimize(neg_log_marginal, np.log([1.0,.35,.1]),
                      args=(X_train,y_train,media_zero),
                      bounds=BOUNDS, method="L-BFGS-B")
params_conv = np.exp(ajuste_conv.x)
mu_conv, sd_conv = gp_predict(X_train,y_train,X_test,params_conv)
print("GP convencional: (sigma_f, ell, sigma_n) =", params_conv)
print("Neg. log-verossimilhança:", ajuste_conv.fun)


In [ ]:
fig, ax = plt.subplots(figsize=(9,4))
ax.scatter(X_train,y_train,marker="x",c="black", label="dados")
ax.plot(X_test, f_real(X_test), "k--", alpha=.65, label="verdade (somente avaliação)")
ax.plot(X_test,mu_conv,label="GP convencional — média posterior")
ax.fill_between(X_test,mu_conv-1.96*sd_conv,mu_conv+1.96*sd_conv,alpha=.2)
ax.set(xlabel="x",ylabel="f(x)",title="GP convencional — hiperparâmetros estimados")
ax.legend(); ax.grid(alpha=.2); plt.show()


## 7. GP estruturado — a hipótese entra na média

Agora o identificador passa a conhecer **apenas a classe de funções**:
\[
m_\phi(x)=
\begin{cases}
ax,&x\le t,\\
bx,&x>t,
\end{cases}
\qquad \phi=(a,b,t).
\]
O identificador continua sem conhecer \(a=1\), \(b=-1\) ou \(t=1\). O GP residual usa o **mesmo tipo de kernel**.

**Um cuidado do notebook original:** os priors lognormais usados para \(b\) atribuem probabilidade zero a \(b<0\), embora a inclinação verdadeira seja negativa; além disso, a execução salva do notebook contém um erro de chave na etapa estruturada. Nesta reprodução estimamos as inclinações **com sinal livre** por máxima verossimilhança.

Como a localização do salto é não suave, testamos limiares entre entradas observadas e, para cada candidato, otimizamos as inclinações e os hiperparâmetros. Nenhuma variável recebe diretamente o limiar verdadeiro.


In [ ]:
def media_piecewise(x, a, b, t):
    x = np.asarray(x,dtype=float).reshape(-1)
    return np.where(x <= t, a*x, b*x)

ordenados = np.sort(X_train)
candidatos = (ordenados[:-1] + ordenados[1:])/2
perfis = []
for t in candidatos:
    H = np.column_stack([np.where(X_train <= t,X_train,0),
                         np.where(X_train > t,X_train,0)])
    ab0 = np.linalg.lstsq(H, y_train, rcond=None)[0]
    x0 = np.r_[ab0, ajuste_conv.x]
    def objetivo(v):
        mean_fn = lambda x: media_piecewise(x,v[0],v[1],t)
        return neg_log_marginal(v[2:],X_train,y_train,mean_fn)
    opt = minimize(objetivo,x0,method="L-BFGS-B",
                   bounds=[(-5,5),(-5,5)]+BOUNDS,options={"maxiter":150})
    perfis.append((float(opt.fun), float(t), opt.x))

melhor = min(perfis,key=lambda z:z[0])
nlml_str,t_hat,v_hat = melhor
a_hat,b_hat = v_hat[:2]
params_str = np.exp(v_hat[2:])
media_str = lambda x: media_piecewise(x,a_hat,b_hat,t_hat)
mu_str,sd_str = gp_predict(X_train,y_train,X_test,params_str,media_str)
print(f"Inclinações estimadas: a={a_hat:.4f}, b={b_hat:.4f}")
print(f"Limiar escolhido: t={t_hat:.4f}")
print("Hiperparâmetros:",params_str)
print("Perfil NLML estruturado:",nlml_str)


In [ ]:
fig, axes = plt.subplots(1,2,figsize=(12,4))
for ax,mu,sd,name in zip(axes,[mu_conv,mu_str],[sd_conv,sd_str],
                         ["convencional","estruturado"]):
    ax.plot(X_test,f_real(X_test),"k--",label="função real")
    ax.plot(X_test,mu,label=f"GP {name}")
    ax.scatter(X_train,y_train,marker="x",c="black",s=30,label="observações")
    ax.fill_between(X_test,mu-1.96*sd,mu+1.96*sd,alpha=.18)
    ax.set(xlim=(0,2),xlabel="x",ylabel="f(x)",title=name.title())
    ax.grid(alpha=.2); ax.legend(fontsize=8)
plt.tight_layout();plt.show()

plt.figure(figsize=(9,3))
plt.plot([z[1] for z in perfis],[z[0] for z in perfis],"o-")
plt.axvline(t_hat,linestyle="--",label="limiar estimado")
plt.xlabel("Candidato de limiar t"); plt.ylabel("Neg. log-verossimilhança")
plt.title("O que as 15 observações permitem dizer sobre o limiar?")
plt.legend();plt.grid(alpha=.2);plt.show()


## 8. Avaliação — erro médio não é toda a história

Como temos uma planta **simulada**, podemos comparar as curvas estimadas à função verdadeira **somente depois de treiná-las**. Medimos MAE, RMSE e RMSE fora da vizinhança do salto.

O GP convencional tem média posterior contínua, portanto não pode representar exatamente a descontinuidade. O estruturado permite um salto na média, mas a posição do salto **não é identificável exatamente** a partir de observações finitas: limiares dentro de um intervalo sem pontos produzem a mesma separação dos dados de treino.

As bandas exibidas são intervalos gaussianos pontuais **condicionais aos hiperparâmetros estimados**, não garantias uniformes nem calibração demonstrada.


In [ ]:
verdade = f_real(X_test)
fora_salto = np.abs(X_test - 1.0) > 0.12
def metricas(mu):
    err = mu-verdade
    return np.sqrt(np.mean(err**2)),np.mean(np.abs(err)),np.sqrt(np.mean(err[fora_salto]**2))
for nome,mu in [("Convencional",mu_conv),("Estruturado",mu_str)]:
    rmse,mae,rmse_fora=metricas(mu)
    print(f"{nome:13s} RMSE={rmse:.4f}  MAE={mae:.4f}  RMSE (fora do salto)={rmse_fora:.4f}")

plt.figure(figsize=(9,4))
plt.plot(X_test,np.abs(mu_conv-verdade),label="erro absoluto — convencional")
plt.plot(X_test,np.abs(mu_str-verdade),label="erro absoluto — estruturado")
plt.axvline(1,linestyle="--",c="black",alpha=.5)
plt.xlabel("x");plt.ylabel("|estimativa − função real|")
plt.title("Erros concentrados na descontinuidade")
plt.grid(alpha=.2);plt.legend();plt.show()


## 9. Conclusões e perguntas de pesquisa

1. O kernel define a correlação e a regularidade das funções candidatas. A média carrega uma hipótese anterior sobre seu comportamento.
2. No GP convencional, a função posterior corrige a média nula com base nas observações.
3. No GP estruturado, uma forma por partes conhecida pode melhorar a previsão com poucos dados, **mas só se a hipótese estrutural estiver correta**.
4. Um erro de limiar muito pequeno pode gerar grande erro quadrático por causa do salto: compare **MAE, RMSE e o erro por região**.
5. A incerteza do GP não é um certificado de correção do modelo; o modelo pode estar mal especificado, especialmente perto do salto.

**Limitações desta reprodução:** kernel Matérn 3/2 escolhido explicitamente; hiperparâmetros por MLE, não por MCMC/HMC; variância posterior ignora a incerteza da estimação dos hiperparâmetros e do limiar. Os números não devem ser tratados como resultados quantitativos publicados pelos autores.

**Próxima etapa:** [GP_Duffing.ipynb](GP_Duffing.ipynb). Lá estimaremos uma **transição dinâmica** \(x_k,u_k\mapsto x_{k+1}\), sem revelar a equação diferencial ao identificador.
